# ShiftLog-Gym Smoke Test + Baselines

This notebook installs the repo, instantiates the environment with `seed=42`, runs the random, scripted, and untrained-LLM baselines, and writes observatory artifacts.

In [ ]:
import os

REPO_URL = "https://github.com/Chirag0096/ShiftLog-Gym.git"
REPO_DIR = "ShiftLog-Gym"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pip -q install -e . pandas matplotlib transformers accelerate

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from shiftlog_gym.scenarios import PUBLIC_FAMILIES
from shiftlog_gym.simulator import ShiftLogSimulator
from shiftlog_gym.training import (
    EpisodeArtifacts,
    summarize_baseline,
    summarize_episode,
    write_artifacts,
    write_episode_replays,
)
from shiftlog_gym.trl_env import ShiftLogToolEnv

OBS_ROOT = Path("observatory")
EPISODES_DIR = OBS_ROOT / "episodes"
TRAINING_RUNS_DIR = OBS_ROOT / "training_runs"
EPISODES_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_RUNS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
env = ShiftLogSimulator()
print(env.reset(seed=SEED, family="db_pool"))

In [ ]:
VALID_TOOLS = [
    "read_shift_log",
    "append_shift_log",
    "update_shift_log",
    "inspect_service",
    "inspect_dependency",
    "run_diagnostic",
    "apply_mitigation",
    "resolve_incident",
]

def random_action(sim, rng):
    incident = sim.active_incident
    if incident is None:
        return None
    tool = rng.choice(VALID_TOOLS)
    if tool == "read_shift_log":
        return tool, {"query": " ".join(incident.relevant_memory_terms[:2]) or incident.service, "limit": 3}
    if tool == "append_shift_log":
        return tool, {
            "entry_type": rng.choice(["fact", "note", "hypothesis"]),
            "incident_id": incident.incident_id,
            "service": incident.service,
            "fact": rng.choice(incident.symptoms),
            "confidence": round(rng.random(), 2),
        }
    if tool == "update_shift_log":
        entries = sim.episode_state.shift_log_entries if sim.episode_state else []
        if not entries:
            return "inspect_service", {"service": incident.service}
        return tool, {"memory_id": entries[-1].memory_id, "patch": incident.summary, "reason": "random-baseline"}
    if tool == "inspect_service":
        return tool, {"service": incident.service}
    if tool == "inspect_dependency":
        return tool, {"service": incident.service}
    if tool == "run_diagnostic":
        diagnostic = next(iter(incident.diagnostics.keys()))
        return tool, {"service": incident.service, "diagnostic": diagnostic}
    if tool == "apply_mitigation":
        choice = rng.choice(incident.valid_mitigations + incident.unsupported_resolutions)
        return tool, {"service": incident.service, "mitigation": choice}
    return tool, {"incident_id": incident.incident_id, "resolution": "invalid_mitigation", "root_cause": "unknown"}

def run_random_baseline(num_episodes=50):
    rng = random.Random(SEED)
    rows, memory_rows, tool_rows = [], [], []
    replay_artifacts = []
    for episode_idx in range(num_episodes):
        family = PUBLIC_FAMILIES[episode_idx % len(PUBLIC_FAMILIES)]
        variant_index = episode_idx % 8
        sim = ShiftLogSimulator()
        sim.reset(seed=episode_idx + 1, family=family, variant_index=variant_index)
        while not sim.done and sim.episode_state.step_count < 40:
            action = random_action(sim, rng)
            if action is None:
                break
            getattr(sim, action[0])(**action[1])
        artifacts = summarize_episode(sim, f"random-{episode_idx:03d}", "baseline", episode_idx + 1, variant_index)
        rows.append(artifacts.episode_row)
        memory_rows.extend(artifacts.memory_events)
        tool_rows.extend(artifacts.tool_timeline)
        replay_artifacts.append(artifacts)
    write_artifacts(OBS_ROOT, rows, memory_rows, tool_rows)
    write_episode_replays(EPISODES_DIR, replay_artifacts)
    summary = summarize_baseline(rows)
    summary["episodes"] = num_episodes
    (OBS_ROOT / "baselines_random.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    random_df = pd.DataFrame(rows)
    print(summary)
    random_df.head()

In [ ]:
def run_scripted_episode(sim):
    while not sim.done:
        incident = sim.active_incident
        if incident is None:
            break
        sim.read_shift_log(" ".join(incident.relevant_memory_terms[:3]) or incident.service, limit=3)
        sim.inspect_service(incident.service)
        diagnostic = next(iter(incident.diagnostics.keys()))
        sim.run_diagnostic(incident.service, diagnostic)
        for _, fact in incident.golden_memory[:1]:
            sim.append_shift_log("fact", incident.incident_id, incident.service, fact, 0.95)
        sim.apply_mitigation(incident.service, incident.resolution)
        sim.resolve_incident(incident.incident_id, incident.resolution, incident.root_cause)

def run_scripted_baseline(num_episodes=50):
    rows, memory_rows, tool_rows = [], [], []
    replay_artifacts = []
    for episode_idx in range(num_episodes):
        family = PUBLIC_FAMILIES[episode_idx % len(PUBLIC_FAMILIES)]
        variant_index = episode_idx % 8
        sim = ShiftLogSimulator()
        sim.reset(seed=episode_idx + 101, family=family, variant_index=variant_index)
        run_scripted_episode(sim)
        artifacts = summarize_episode(sim, f"scripted-{episode_idx:03d}", "baseline", episode_idx + 101, variant_index)
        rows.append(artifacts.episode_row)
        memory_rows.extend(artifacts.memory_events)
        tool_rows.extend(artifacts.tool_timeline)
        replay_artifacts.append(artifacts)
    write_episode_replays(EPISODES_DIR, replay_artifacts)
    summary = summarize_baseline(rows)
    summary["episodes"] = num_episodes
    (OBS_ROOT / "baselines_scripted.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    scripted_df = pd.DataFrame(rows)
    print(summary)
    scripted_df.head()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

RUN_LLM_BASELINE = False
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

def parse_action(raw_text, incident):
    try:
        block = raw_text[raw_text.index("{") : raw_text.rindex("}") + 1]
        payload = json.loads(block)
        return payload.get("tool", "inspect_service"), payload.get("arguments", {"service": incident.service})
    except Exception:
        return "inspect_service", {"service": incident.service}

def run_llm_baseline(num_episodes=20):
    if not RUN_LLM_BASELINE:
        skipped = {"episodes": 0, "note": "Set RUN_LLM_BASELINE=True to execute the base model baseline."}
        (OBS_ROOT / "baselines_llm_base.json").write_text(json.dumps(skipped, indent=2), encoding="utf-8")
        return pd.DataFrame()
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, torch_dtype="auto", device_map="auto")
    rows, replay_artifacts = [], []
    for episode_idx in range(num_episodes):
        family = PUBLIC_FAMILIES[episode_idx % len(PUBLIC_FAMILIES)]
        variant_index = episode_idx % 8
        env = ShiftLogToolEnv(rollout_mode="full")
        observation = env.reset(seed=episode_idx + 201, family=family, variant_index=variant_index)
        sim = env.simulator
        transcript = []
        for _ in range(18):
            if env.done:
                break
            incident = sim.active_incident
            prompt = (
                observation
                + "\nReturn a compact JSON object with keys tool and arguments. "
                + "Prefer reading the shift log before mitigation."
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model.generate(**inputs, max_new_tokens=120, do_sample=False)
            text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            tool, arguments = parse_action(text, incident)
            if tool not in VALID_TOOLS:
                tool, arguments = "inspect_service", {"service": incident.service}
            observation = getattr(env, tool)(**arguments)
            transcript.append({"tool": tool, "arguments": arguments, "observation": observation})
        artifacts = summarize_episode(sim, f"llm-base-{episode_idx:03d}", "baseline", episode_idx + 201, variant_index)
        rows.append(artifacts.episode_row)
        replay_artifacts.append(artifacts)
    write_episode_replays(EPISODES_DIR, replay_artifacts)
    summary = summarize_baseline(rows)
    summary["episodes"] = num_episodes
    (OBS_ROOT / "baselines_llm_base.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return pd.DataFrame(rows)

random_df = run_random_baseline(50)
scripted_df = run_scripted_baseline(50)
llm_df = run_llm_baseline(20)

In [ ]:
random_summary = json.loads((OBS_ROOT / "baselines_random.json").read_text(encoding="utf-8"))
scripted_summary = json.loads((OBS_ROOT / "baselines_scripted.json").read_text(encoding="utf-8"))
llm_summary = json.loads((OBS_ROOT / "baselines_llm_base.json").read_text(encoding="utf-8"))

baseline_table = pd.DataFrame([
    {"agent": "Random Agent", **random_summary},
    {"agent": "Scripted Agent", **scripted_summary},
    {"agent": "Untrained LLM", **llm_summary},
])

plot_columns = [
    "recall_before_action_rate",
    "linked_incident_success_rate",
    "noise_resistance_rate",
    "avg_total_reward",
]
ax = baseline_table.set_index("agent")[plot_columns].plot(kind="bar", figsize=(12, 5))
ax.set_title("ShiftLog-Gym Baseline Comparison")
ax.set_ylabel("Score")
ax.figure.tight_layout()
plt.savefig(OBS_ROOT / "baseline_comparison.png", dpi=200)
plt.show()

baselines_payload = {
    "random": random_summary,
    "scripted": scripted_summary,
    "llm_base": llm_summary,
    "trained_llm": {},
}
(OBS_ROOT / "baselines.json").write_text(json.dumps(baselines_payload, indent=2), encoding="utf-8")
baseline_table

In [ ]:
gap = scripted_summary.get("recall_before_action_rate", 0.0) - random_summary.get("recall_before_action_rate", 0.0)
print("Scripted vs random recall-before-action gap:", round(gap, 4))
baseline_table[["agent", "recall_before_action_rate"]]